### SDP: eddy-wide pigment summary

In [ ]:
import sys
from typing import cast
from pathlib import Path

ROOT = Path.cwd().parent.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from utils.config import resolve_output_dir

EXPERIMENT = "bgc_20241001_20250701"
TRACK_ID = 13
DATE = pd.Timestamp("2025-01-28")

# Diagnostic biomarker pigments (T chla shown separately as annotation since it dominates the scale)
PIGMENT_NAMES = ["DV chla", "MV chlb", "chl c1+c2", "chl c3",
    "Fuco", "ButFuco", "HexFuco", "Perid", "Allo", "Zea", "Viola", "Neo"]

PIGMENT_DISPLAY = [
    "DV Chl-a", "MV Chl-b", "Chl-c1+c2", "Chl-c3",
    "Fucoxanthin", "But-Fuco", "Hex-Fuco", "Peridinin",
    "Alloxanthin", "Zeaxanthin", "Violaxanthin", "Neoxanthin",
]

# Each biomarker pigment is colored by its primary PFT association
PIGMENT_COLORS = [
    "#1565C0",  # DV chla -> Cyanobacteria (Prochlorococcus)
    "#2E7D32",  # MV chlb -> Green algae
    "#FFB347",  # chl c1+c2 -> Haptophytes
    "#FFB347",  # chl c3 -> Haptophytes
    "#E87A14",  # Fuco -> Diatoms
    "#FFB347",  # ButFuco -> Haptophytes
    "#FFB347",  # HexFuco -> Haptophytes
    "#C62828",  # Perid -> Dinoflagellates
    "#AD1457",  # Allo -> Cryptophytes
    "#1565C0",  # Zea -> Cyanobacteria
    "#2E7D32",  # Viola -> Green algae
    "#2E7D32",  # Neo -> Green algae
]

In [ ]:
pig = pd.read_parquet(resolve_output_dir(EXPERIMENT, "pigments", "cyclone") / f"eddy_{TRACK_ID}_pigments.parquet")

pig_d = pig[pig["date"] == DATE].reset_index(drop=True)
n_pixels = len(pig_d)

pig_mean = pig_d[PIGMENT_NAMES].mean().to_numpy()  # pyright: ignore[reportAttributeAccessIssue]
pig_std = pig_d[PIGMENT_NAMES].std().to_numpy()  # pyright: ignore[reportAttributeAccessIssue]
total_chla_mean = pig_d["T chla"].mean()

print(f"pixels: {n_pixels}")
print(f"mean_total_chla_mg_m3: {total_chla_mean:.3f}")


In [ ]:
fig = cast(Figure, plt.figure(figsize=(7, 3.8), dpi=300))
ax1 = fig.add_subplot(111)

x_pos = np.arange(len(PIGMENT_NAMES))
ax1.bar(x_pos, pig_mean, yerr=pig_std, color=PIGMENT_COLORS,
    edgecolor="0.25", linewidth=0.5,
    error_kw=dict(ecolor="0.3", lw=0.8, capsize=2.5, capthick=0.8))
ax1.set_xticks(x_pos)
ax1.set_xticklabels(PIGMENT_DISPLAY, rotation=45, ha="right", fontsize=8)
ax1.set_ylabel(r"Concentration (mg m$^{-3}$)", fontsize=9)
ax1.tick_params(labelsize=8)
ax1.spines[["top", "right"]].set_visible(False)
ax1.set_title(f"Cyclone {TRACK_ID}: diagnostic pigments on {DATE:%Y-%m-%d}", fontsize=10, pad=6)
ax1.text(0.98, 0.95, f"Total Chl-$a$ = {total_chla_mean:.2f} mg m$^{{-3}}$",
    transform=ax1.transAxes, fontsize=8, ha="right", va="top", color="0.3",
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="0.7", linewidth=0.5))

fig.savefig("sdp.png", dpi=300, bbox_inches="tight")
plt.show()